# EDA and type correction: cleaning a sales CSV file

**EDA** (Exploratory Data Analysis) means looking at the data first, to find its problems before we use it.

### Requirements
1. Read the data from the sales sample CSV file and analyse it to identify problems
2. Fix the problems: give every column the right type and a clear name, and remove the bad rows

### Steps
1. Define the schema
2. Read the file
3. Explore the data and list the problems
4. Fix the types and clean the data
5. Check the result

## Step 1. Define the schema

Best practice: **define a schema before reading the file** (schema on read, notebook 07).
It tells Spark each column's type, so nothing is guessed.

There are two ways to write a schema:

| Approach | Example | When to use |
|---|---|---|
| `StructType` / `StructField` objects | `StructType([StructField("id", IntegerType()), ...])` | When the schema is built or changed with Python code |
| **DDL-formatted string** | `"id int, name string, ..."` | Most cases: shorter and easier to read (same format as `CREATE TABLE`) |

**Why are `dop`, `amount` and `discount` read as `string`?**
These columns hold messy values (dates in different formats, the text `nil`, a number too big for an `int`).
Reading them as `string` keeps the **raw values exactly as they are in the file**, so no data is lost at read time.
We fix the types afterwards, in step 4, where we control how each bad value is handled.

In [2]:
# Define the schema first, so every column gets an explicit type and nothing is inferred.
# This is the best practice: define a schema before reading the data into a DataFrame.

# Approach 1: StructType and StructField classes from pyspark.sql.types (same schema as below):
#   from pyspark.sql.types import StructType, StructField, IntegerType, StringType, LongType
#   file_schema = StructType([
#       StructField("id", IntegerType()),
#       StructField("name", StringType()),
#       ...
#   ])

# Approach 2 (used here): a DDL-formatted string, "column_name type, column_name type, ..."
# dop, amount and discount are read as string on purpose: they hold messy values,
# so we keep them raw and correct their types later.
file_schema = """
id int,
name string,
dop string,
phone long,
amount string,
discount string
"""

## Step 2. Read the file

The CSV connector with `header = true` (first line = column names) and our schema.

Normally, for a date column we would read it as `date` and give its pattern with `.option("dateFormat", ...)`.
Here that doesn't work: `dop` has **dates in different formats**, and one `dateFormat` can only parse one of them.
So we read it as `string` and parse it in step 4.

In [3]:
# Normally a date column is read as date, with .option("dateFormat", "...") giving its pattern.
# Here dop has dates in more than one format, so we read it as a string and parse it later.
sales_raw_df = (
    spark.read.format("csv")
        .option("header", "true")   # first line = column names
        .schema(file_schema)        # our schema, no inference
        .load("/Volumes/dev/spark_db/datasets/spark_programming/data/sales_sample.csv")
)

# Look at the raw data
display(sales_raw_df)

,id,name,dop,phone,amount,discount
0,100,Prashant,2020-06-15,9.238615e+09,12000,18.5
1,101,David,2018-08-7,8.908618e+09,15000,nil
2,102,Simran,14-05-2019,NaN,3000000000,21


## Step 3. Explore the data: find the problems

`describe()` computes summary statistics for each column: `count`, `mean`, `stddev`, `min`, `max`.
It's a quick way to spot missing values and suspicious numbers.

How to read it:
- `count` counts **non-null** values: a column with a lower count has missing values.
- On `string` columns, `mean` and `stddev` use only the values that look like numbers, and `min` / `max` are compared **as text** (alphabetically).
  That's why the `max` of `discount` is `nil`: letters sort after digits.

In [4]:
# Summary statistics of the raw data: count, mean, stddev, min, max per column
display(sales_raw_df.describe())

,summary,id,name,dop,phone,amount,discount
0,count,3,3,3,2,3,3
1,mean,101.0,None,None,9.0736163E9,1.000009E9,19.75
2,stddev,1.0,None,None,2.3334338517179397E8,1.7320430133408928E9,1.7677669529663689
3,min,100,David,14-05-2019,8908617610,12000,18.5
4,max,102,Simran,2020-06-15,9238614990,3000000000,nil


### Problems found

| Column | Problem | Example |
|---|---|---|
| `id` | An identifier, not a number to calculate with | `100` |
| `dop` (date of purchase) | Dates in **different formats**, stored as text | `2020-06-15`, `2018-08-7`, `14-05-2019` |
| `phone` | One phone number is **missing** (`count` = 2). It is also an identifier, not a number | `9238614990` |
| `amount` | An **outlier**: far bigger than the others (and too big for an `int`) | `3000000000` |
| `discount` | A **non-numeric value**, so the column can't be a number as is | `nil` |
| all | Short, unclear column names | `dop` |

> In VS Code, `display()` shows `phone` as `9.238615e+09` and the missing value as `NaN`.
> That's only how the table is drawn (a whole-number column with a missing value is shown as decimals). The data is a `long` with a `null`.

## Step 4. Fix the types and clean the data

We fix every column in one `selectExpr()`, then remove the bad rows with `filter()`.

- `select("col")` only takes column names. `selectExpr("sql expression")` takes **SQL expressions**, so we can cast, call functions and rename (`as`) in the same step.
- `filter("condition")` keeps the rows that match the condition. It is the same as `where()`.

| New column | Expression | What it fixes |
|---|---|---|
| `Transaction_ID` | `cast(id as string)` | An ID is a label, not a number to add up |
| `Customer_Name` | `name` | Only renamed |
| `Date_of_Purchase` | `nvl(try_cast(dop as date), to_date(dop, 'dd-MM-yyyy'))` | Parses both date formats (see below) |
| `Customer_Phone` | `cast(phone as string)` | A phone number is a label (it could also start with `0` or `+`) |
| `Purchase_Amount` | `cast(amount as long)` | `long`, because `3000000000` doesn't fit in an `int` |
| `Applied_Discount` | `nvl(try_cast(discount as double), 0)` | `nil` becomes `0` |

The functions used:
- **`cast(x as type)`**: convert to another type. If the value can't be converted, it fails (with ANSI mode on, as in Databricks).
- **`try_cast(x as type)`**: the same, but it returns **`null`** instead of failing. Perfect for messy values like `nil`.
- **`nvl(a, b)`**: returns `a`, or `b` if `a` is `null` (same as `coalesce(a, b)`).
- **`to_date(x, 'pattern')`**: parse text into a date with a given pattern.

**How the date expression works:**
1. `try_cast(dop as date)` parses the standard `yyyy-MM-dd` format (`2020-06-15`, and also `2018-08-7`).
   For `14-05-2019` it can't, so it returns `null`.
2. When the first result is `null`, `nvl` uses the second one: `to_date(dop, 'dd-MM-yyyy')`, which parses `14-05-2019`.

**The filter:** `Purchase_Amount is not null and Purchase_Amount < 200000` removes rows with no amount, and the outlier.
The `200000` limit is a business rule: we assume no real purchase is that big, so that value is a data error.

In [14]:
# select() only takes column names; selectExpr() takes SQL expressions,
# so we can cast, call functions and rename ("as") each column in one step.
sales_df = sales_raw_df.selectExpr(

    # IDs and phone numbers are labels, not numbers to calculate with -> string
    "cast(id as string) as Transaction_ID",
    "name as Customer_Name",

    # The dates come in 2 formats: yyyy-MM-dd and dd-MM-yyyy.
    # 1. try_cast(dop as date) parses yyyy-MM-dd; for dd-MM-yyyy it returns null instead of failing
    # 2. nvl(a, b) returns b when a is null -> then to_date(dop, 'dd-MM-yyyy') parses the other format
    "nvl(try_cast(dop as date), to_date(dop, 'dd-MM-yyyy')) as Date_of_Purchase",

    "cast(phone as string) as Customer_Phone",

    # long, not int: 3000000000 is bigger than the int maximum (about 2.1 billion)
    "cast(amount as long) as Purchase_Amount",

    # discount has non-numeric values like "nil", so a plain cast to double would fail.
    # try_cast returns null for them, and nvl(..., 0) replaces that null with 0 (no discount),
    # so we can still calculate the final amount after discount.
    "nvl(try_cast(discount as double), 0) as Applied_Discount"

# filter() is the same as where(): keep only the rows that match the condition.
# Here: drop rows with no amount, and the outlier (business rule: no purchase is 200000 or more).
).filter("Purchase_Amount is not null and Purchase_Amount < 200000")

display(sales_df)

,Transaction_ID,Customer_Name,Date_of_Purchase,Customer_Phone,Purchase_Amount,Applied_Discount
0,100,Prashant,2020-06-15,9238614990,12000,18.5
1,101,David,2018-08-07,8908617610,15000,0.0


## Step 5. Check the result

Run `describe()` again, this time on the **cleaned** DataFrame, and compare with step 3:
the columns now have real types, the outlier and `nil` are gone, and the statistics make sense.

> Note: the outlier row (`Simran`, amount `3000000000`) is removed by the filter.
> It is also the row with the `14-05-2019` date, so the `dd-MM-yyyy` part of the date expression isn't visible in the final result.
> To check it, run `display(sales_df)` without the `.filter(...)`.

In [ ]:
# Summary statistics of the CLEANED data, to compare with the raw data in step 3
display(sales_df.describe())